# Async resources and durable jobs

This example requires SDK0.2. Default execution uses synthetic transport, not an FM. Explicit live mode uses the qualified bounded classifier. Async calls and durable background jobs are different: await starts/waits for a request, while the job identity supports reconnect after local timeout/cancellation.

In [ ]:
from support import notebook_async_client, synthetic_table
columns, train, labels, query = synthetic_table()

In [ ]:
async with notebook_async_client() as client:
    model = next(m for m in await client.models() if m["id"] == "tabicl-v2")
    assert model["status"] == "available" and "classification" in model["tasks"]
    dataset = await client.upload(columns=columns+["label"], rows=[r.tolist()+[y] for r,y in zip(train,labels)], target="label")
    job = await client.submit_fit(dataset["id"], model="tabicl-v2", task="classification", seed=9)
    durable_id = job.id
    reconnected = client.job(durable_id)  # factory is not awaited
    predictor = await reconnected.result(timeout=120)
    result = await client.predict(predictor["id"], columns=columns, rows=query.tolist())
    assert len(result["predictions"]) == 32
    assert result["model_version"] == predictor["model_version"]

The async context closes its HTTP client. Cancelling a local await or hitting a local wait deadline leaves durable work intact. Explicit await job.cancel() requests remote cancellation; subsequent result retrieval still requires current workspace authorization. GET-only bounded retries do not replay POST fits or uploads. Batch/resumable files and causal async operations remain unreleased gates.